<a href="https://colab.research.google.com/github/KsiuTretyakova/MachineLearning/blob/main/%D0%9A%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F_%D0%B7%D0%BE%D0%B1%D1%80%D0%B0%D0%B6%D0%B5%D0%BD%D1%8C_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Класифікація зображень CIFAR-10 за допомогою CNN

CNN (Convolutional Neural Network) — згорткова нейронна мережа. Вона автоматично виділяє ознаки (features) із зображення та навчається класифікувати їх.

#1. Імпортуємо необхідні бібліотеки
*Ці бібліотеки вже попередньо встановлені в Google Colab*

In [ ]:
# TensorFlow — бібліотека для машинного навчання
import tensorflow as tf
# Keras — високорівнева API для створення нейронних мереж
# layers — модуль із типами шарів нейромереж (наприклад, Conv2D, MaxPooling2D тощо)
# models — модуль для створення моделі
from tensorflow.keras import layers, models
# Matplotlib — для побудови графіків
import matplotlib.pyplot as plt
# NumPy — робота з масивами
import numpy as np
# OpenCV (cv2) — бібліотека для обробки зображень
import cv2
# PIL — для роботи із зображеннями
from PIL import Image

#2. Завантаження та підготовка набору CIFAR-10

CIFAR-10 — набір зображень розміром 32x32 пікселі в 10 класах

['літак', 'автомобіль', 'птах', 'кіт', 'олень', 'собака', 'жаба', 'кінь', 'корабель', 'вантажівка']

In [ ]:
# Завантаження CIFAR-10 (32x32 зображення у 10 класах)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
print(X_test)
print(y_test)

[[[[158 112  49]
   [159 111  47]
   [165 116  51]
   ...
   [137  95  36]
   [126  91  36]
   [116  85  33]]

  [[152 112  51]
   [151 110  40]
   [159 114  45]
   ...
   [136  95  31]
   [125  91  32]
   [119  88  34]]

  [[151 110  47]
   [151 109  33]
   [158 111  36]
   ...
   [139  98  34]
   [130  95  34]
   [120  89  33]]

  ...

  [[ 68 124 177]
   [ 42 100 148]
   [ 31  88 137]
   ...
   [ 38  97 146]
   [ 13  64 108]
   [ 40  85 127]]

  [[ 61 116 168]
   [ 49 102 148]
   [ 35  85 132]
   ...
   [ 26  82 130]
   [ 29  82 126]
   [ 20  64 107]]

  [[ 54 107 160]
   [ 56 105 149]
   [ 45  89 132]
   ...
   [ 24  77 124]
   [ 34  84 129]
   [ 21  67 110]]]


 [[[235 235 235]
   [231 231 231]
   [232 232 232]
   ...
   [233 233 233]
   [233 233 233]
   [232 232 232]]

  [[238 238 238]
   [235 235 235]
   [235 235 235]
   ...
   [236 236 236]
   [236 236 236]
   [235 235 235]]

  [[237 237 237]
   [234 234 234]
   [234 234 234]
   ...
   [235 235 235]
   [235 235 235]
   [234 234

In [ ]:
# Нормалізація — перетворення значень пікселів у діапазон [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# Список назв класів українською мовою
class_name = ['літак', 'автомобіль', 'птах', 'кіт', 'олень', 'собака', 'жаба', 'кінь', 'корабель', 'вантажівка']

#3. Побудова моделі CNN

CNN (Convolutional Neural Network) — згорткова нейронна мережа, що добре працює з зображеннями

- Conv2D: згортковий шар, що виявляє особливості (features)
- MaxPooling2D: зменшує розмірність, зберігаючи важливі ознаки
- Flatten: вирівнює багатовимірні дані у вектор
- Dense: повнозв'язний шар

CNN (Convolutional Neural Network) — працює як ієрархія ознак:
- перші шари шукають прості шаблони (краї, кольорові переходи),
- середні — формують структури (кут, текстура),
- глибші — бачать складні форми (вухо кота, крило літака тощо).

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),  # перший згортковий шар

    # 32 фільтри (або ядра згортки) розміром 3x3
    # relu (Rectified Linear Unit) — нелінійна функція активації: f(x) = max(0, x)
    # input_shape=(32, 32, 3) — 32x32 пікселі, 3 кольорові канали (RGB)

    layers.MaxPooling2D((2,2)),  # 32x32 -> 16x16  # підвибірка (pooling), зменшення розміру у 2 рази

    layers.Conv2D(64, (3, 3), activation='relu'),  # другий згортковий шар
    layers.MaxPooling2D((2,2)),  # 16x16 -> 8x8  # ще одна підвибірка

    layers.Conv2D(64, (3, 3), activation='relu'),  # третій згортковий шар

    layers.Flatten(),  # перетворення вектора ознак
    layers.Dense(64, activation='relu'),  # прихований повнозв’язний шар

    # Вихідний шар з 10 нейронами (по одному на кожен клас)
    # Softmax — функція, що перетворює числа у ймовірності (сумуються до 1.0)
    layers.Dense(10, activation='softmax')  # вихідний шар: 10 класів
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


| Термін                             | Пояснення                                                              |
| ---------------------------------- | ---------------------------------------------------------------------- |
| **Convolutional Layer** *(Conv2D)* | Згортковий шар — виділяє локальні ознаки на зображеннях                |
| **Pooling Layer** *(MaxPooling2D)* | Зменшує розмірність, залишаючи найважливіші особливості                |
| **Flatten**                        | Перетворює матрицю ознак у плоский вектор                              |
| **Dense Layer**                    | Повнозв’язний шар — кожен нейрон з’єднаний з усіма з попереднього шару |
| **Softmax**                        | Перетворює вихід на ймовірності кожного класу                          |
| **Epoch**                          | Один прохід усіх тренувальних даних через модель                       |


#4. Компіляція моделі

- optimizer='adam': адаптивний оптимізатор Adam — оптимізатор, що комбінує методи моментуму та адаптивного навчання
- loss='sparse_categorical_crossentropy': функція втрат для багатокласової класифікації (мітки у вигляді чисел, не one-hot)
- metrics=['accuracy']: точність як метрика - скільки разів модель «бачить» усі тренувальні дані

In [ ]:
model.compile(
    optimizer='adam',  # оптимізатор, що адаптивно навчає ваги
    loss='sparse_categorical_crossentropy',  # функція втрат для багатокласової задачі
    metrics=['accuracy']  # точність — основна метрика
)

#5. Навчання моделі (тренування)

Модель буде проходити 10 епох (повних переглядів всіх даних)

In [ ]:
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.4486 - loss: 1.5135 - val_accuracy: 0.5303 - val_loss: 1.2861
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5879 - loss: 1.1664 - val_accuracy: 0.5991 - val_loss: 1.1541
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.6476 - loss: 1.0082 - val_accuracy: 0.6094 - val_loss: 1.1065
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6813 - loss: 0.9112 - val_accuracy: 0.6647 - val_loss: 0.9643
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7070 - loss: 0.8381 - val_accuracy: 0.6884 - val_loss: 0.9103
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7232 - loss: 0.7883 - val_accuracy: 0.6884 - val_loss: 0.8932
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7411 - loss: 0.7380 - val_accuracy: 0.7037 - val_loss: 0.8760
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7568 - loss: 0.6957 -

#6. Оцінка точності моделі на тестових даних